# Kronos LoRA Fine-Tuned – Evaluation (5 Seeds)

Evaluates the fine-tuned **NeoQuasar/Kronos-base** model on energy-sector test data (2021+).  
Parameters are held identical to the zero-shot baseline for scientific comparability.

| Parameter | Value | Note |
|---|---|---|
| `context_steps` | 80 | **Matches zero-shot baseline** |
| `forecast_steps` | 12 | **Matches zero-shot baseline** |
| `stride_steps` | 12 | **Matches zero-shot baseline** |
| `steps` | 120 | **Matches zero-shot baseline** |
| Seeds | 13, 42, 123, 456, 789 | **Matches zero-shot baseline** |
| Test period | 2021-01-01 → today | **Matches zero-shot baseline** |
| `lora_r` | 8 | Optimal from sensitivity analysis |
| `lora_alpha` | 16 | Optimal from sensitivity analysis |
| `lora_dropout` | 0.05 | Optimal from sensitivity analysis |
| `learning_rate` | 1e-4 | Optimal from sensitivity analysis |

**Workflow:**
1. Setup environment
2. Adapter laden (Upload **oder** Re-Training)
3. Evaluation über alle 5 Seeds
4. Cross-Sectional RankIC aggregieren
5. Vergleich Base vs. Fine-tuned
6. Ergebnisse herunterladen

## 1 · Setup: Clone Repository

In [10]:
import os
from pathlib import Path

REPO_NAME = "ba"
REPO_URL  = "https://github.com/bp571/ba.git"

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY
Path(".env").write_text(f"TIINGO_API_KEY={TIINGO_API_KEY}\n")
print("✅ API key configured")

!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("✅ Kronos cloned")

Cloning into 'ba'...
remote: Enumerating objects: 537, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 537 (delta 0), reused 2 (delta 0), pack-reused 529 (from 1)
Receiving objects: 100% (537/537), 13.39 MiB | 20.29 MiB/s, done.
Resolving deltas: 100% (236/236), done.
Working directory: /content/ba/ba
✅ API key configured
Cloning into '02_finetuning/models/Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 371 (delta 6), reused 3 (delta 3), pack-reused 361 (from 2)
Receiving objects: 100% (371/371), 9.32 MiB | 19.59 MiB/s, done.
Resolving deltas: 100% (174/174), done.
✅ Kronos cloned


## 2 · Install Dependencies

In [11]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm einops \
    peft transformers huggingface_hub \
    gluonts python-dotenv tiingo scipy

In [12]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4


## 3 · Adapter laden

**Option A** – Adapter aus vorherigem Training-Notebook hochladen (`adapter_model.safetensors` + `adapter_config.json`)  
**Option B** – Adapter hier neu trainieren (~15 min auf A100)

Führe **nur eine** der beiden folgenden Zellen aus.

In [13]:
# ── OPTION B: Re-train adapter (~15 min auf A100) ─────────────────────────────
!python 02_finetuning/training/train_kronos_lora.py

ADAPTER_PATH = "models/kronos-lora-finetuned/final"
print(f"\nAdapter path: {ADAPTER_PATH}")

Device: cuda

Loading tokenizer: NeoQuasar/Kronos-Tokenizer-base
config.json: 100% 301/301 [00:00<00:00, 1.76MB/s]
model.safetensors: 100% 15.8M/15.8M [00:01<00:00, 13.0MB/s]
Loading model: NeoQuasar/Kronos-base
config.json: 100% 228/228 [00:00<00:00, 1.05MB/s]
model.safetensors: 100% 409M/409M [00:02<00:00, 185MB/s] 

Applying LoRA:
  Rank: 8
  Alpha: 16
  Target modules: 48
trainable params: 638,976 || all params: 102,949,568 || trainable%: 0.6207

Training:
  Max steps: 1000
  Batch size: 4
  Gradient accumulation: 2
  Output: /content/ba/ba/models/kronos-lora-finetuned

Step 10/1000: Loss=3.6985, S1=5.6980, S2=9.0959
Step 20/1000: Loss=3.7347, S1=5.8012, S2=9.1375
Step 30/1000: Loss=3.5146, S1=4.9147, S2=9.1438
Step 40/1000: Loss=3.6490, S1=5.5888, S2=9.0073
Step 50/1000: Loss=3.5979, S1=5.5752, S2=8.8165
Step 60/1000: Loss=3.7139, S1=5.7800, S2=9.0756
Step 70/1000: Loss=3.4846, S1=4.8807, S2=9.0577
Step 80/1000: Loss=3.5806, S1=5.4554, S2=8.8668
Step 90/1000: Loss=3.4658, S1=5.136

In [14]:
# Adapter verifizieren
adapter_path = Path(ADAPTER_PATH)
assert adapter_path.exists(), f"Adapter not found: {adapter_path}"
for f in adapter_path.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
print("✅ Adapter verified")

  README.md  (5 KB)
  adapter_model.safetensors  (2508 KB)
  adapter_config.json  (1 KB)
✅ Adapter verified


## 4 · Evaluation über 5 Seeds

Identische Parameter zum Zero-Shot Baseline: `context=80, forecast=12, stride=12, steps=120`  
Test-Set: 2021-01-01 bis heute.

In [15]:
SEEDS = [13, 42, 123, 456, 789]

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"Seed: {seed}")
    print(f"{'='*60}")
    !python 02_finetuning/evaluation/main_kronos_finetuned.py \
        --adapter-path {ADAPTER_PATH} \
        --seed {seed} \
        --context 80 \
        --forecast 12 \
        --config config/assets.yaml


Seed: 13
✅ Random Seeds auf 13 gesetzt für Reproduzierbarkeit
Loading fine-tuned Kronos model from: models/kronos-lora-finetuned/final
config.json: 100% 301/301 [00:00<00:00, 1.55MB/s]
model.safetensors: 100% 15.8M/15.8M [00:01<00:00, 11.1MB/s]
config.json: 100% 228/228 [00:00<00:00, 1.11MB/s]
model.safetensors: 100% 409M/409M [00:02<00:00, 185MB/s] 
Loading Kronos LoRA adapter: models/kronos-lora-finetuned/final
Loading assets: 100% 34/34 [00:00<00:00, 39.54it/s]
📊 Total windows to process: 3454 across 34 assets
Processing windows:   0% 0/3454 [00:00<?, ?it/s]Processing batch group 1/1: context=80, forecast=12, n_windows=48
Processing windows:   1% 48/3454 [00:02<02:42, 20.93it/s]Processing batch group 1/1: context=80, forecast=12, n_windows=48
Processing windows:   3% 96/3454 [00:04<02:27, 22.78it/s]Processing batch group 1/1: context=80, forecast=12, n_windows=48
Processing windows:   4% 144/3454 [00:06<02:21, 23.39it/s]Processing batch group 1/1: context=80, forecast=12, n_windows

## 5 · Cross-Sectional RankIC aggregieren

Berechnet die RankIC-Statistik über alle Seeds und Assets — exakt dieselbe Methodik wie beim Base-Modell.

In [16]:
!python 01_model_comparison/scripts/evaluate_results.py \
    --results-dir 02_finetuning/results/kronos_finetuned


MULTI-SEED EVALUATION: 02_finetuning/results/kronos_finetuned
Gefundene Seeds: 5
/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)

📊 Final Statistics:
   Seeds:                5
   Total Days:           6120
   Effective N:          911.5199879329688
   Mean RankIC:          0.0254
   95% CI:               [0.0091, 0.0418]
   Standard Deviation:   0.2520
   t-statistic:          3.05
   p-value (H0: IC=0):   0.0024
   Result:               ✅ SIGNIFICANT

📈 Directional Analysis:
   Positive IC Days:     3317/6120 (54.2%)



## 6 · Vergleich: Kronos Base vs. Kronos Fine-tuned

Aggregiert beide Modelle über alle 5 Seeds. Selbe Methodik, selbe Parameter → direkt vergleichbar.

In [17]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr, t as t_dist
import sys
sys.path.insert(0, str(Path.cwd()))
from experiments.metrics import calculate_ic_statistics

SEEDS = [13, 42, 123, 456, 789]


def collect_cross_sectional_rankic(results_base_dir: str, seeds: list) -> list:
    """Lädt raw_values aller Seeds und berechnet cross-sectional RankIC."""
    base_path = Path(results_base_dir)
    all_ic_values = []

    for seed in seeds:
        seed_dir = base_path / f"seed_{seed}"
        final_file = seed_dir / "final_energy_study.json"
        if not final_file.exists():
            print(f"  ⚠️  Missing: {final_file}")
            continue

        with open(final_file) as f:
            data = json.load(f)
        summary = data.get("summary", {})

        actual_dfs, pred_dfs, anchor_dfs = [], [], []

        for ticker in summary:
            res_file = seed_dir / f"result_{ticker}.json"
            if not res_file.exists():
                continue
            with open(res_file) as f:
                res = json.load(f)
            rv = res["raw_values"]
            dates = pd.to_datetime(rv["dates"])
            actual_dfs.append(pd.Series(rv["actual"],    index=dates, name=ticker))
            pred_dfs.append(  pd.Series(rv["predicted"], index=dates, name=ticker))
            if "anchors" in rv:
                anchor_dfs.append(pd.Series(rv["anchors"], index=dates, name=ticker))

        if not actual_dfs:
            continue

        df_act = pd.concat(actual_dfs, axis=1).sort_index()
        df_pre = pd.concat(pred_dfs,   axis=1).sort_index()

        if anchor_dfs:
            df_anc    = pd.concat(anchor_dfs, axis=1).sort_index()
            df_act_r  = np.log(df_act / df_anc)
            df_pre_r  = np.log(df_pre / df_anc)
        else:
            df_act_r = np.log(df_act / df_act.shift(1))
            df_pre_r = np.log(df_pre / df_pre.shift(1))

        df_act_r = df_act_r.dropna(how="all")
        df_pre_r = df_pre_r.dropna(how="all")
        common   = df_act_r.index.intersection(df_pre_r.index)
        df_act_r = df_act_r.loc[common]
        df_pre_r = df_pre_r.loc[common]

        for t in df_act_r.index:
            a_t  = df_act_r.loc[t]
            p_t  = df_pre_r.loc[t]
            mask = a_t.notna() & p_t.notna()
            if mask.sum() >= 10:
                ric, _ = spearmanr(p_t[mask], a_t[mask])
                all_ic_values.append(ric)

    return all_ic_values


def summarise(ic_values: list, label: str) -> dict:
    stats  = calculate_ic_statistics(ic_values, prefix="RankIC")
    mean   = stats["RankIC_Mean"]
    ci_lo, ci_hi = stats["RankIC_CI95"]
    std    = stats["RankIC_Std"]
    n_eff  = stats["RankIC_Effective_N"]
    n_tot  = stats["RankIC_Count"]
    t_stat = mean / (std / np.sqrt(n_eff)) if n_eff > 1 else float("nan")
    p_val  = 2 * (1 - t_dist.cdf(abs(t_stat), df=max(1, n_eff - 1))) if n_eff > 1 else float("nan")
    pos    = sum(v > 0 for v in ic_values)
    return {
        "Model":        label,
        "Mean RankIC":  round(mean, 4),
        "95% CI":       f"[{ci_lo:.4f}, {ci_hi:.4f}]",
        "Std":          round(std, 4),
        "t-stat":       round(t_stat, 2),
        "p-value":      round(p_val, 4),
        "Positive Days": f"{pos}/{n_tot} ({pos/n_tot*100:.1f}%)",
        "N (days)": n_tot,
    }


print("Loading Base-Model results …")
ic_base = collect_cross_sectional_rankic("01_model_comparison/results/kronos", SEEDS)

print("Loading Fine-tuned results …")
ic_ft   = collect_cross_sectional_rankic("02_finetuning/results/kronos_finetuned", SEEDS)

rows = []
if ic_base:
    rows.append(summarise(ic_base, "Kronos Base (zero-shot)"))
else:
    print("⚠️  No base-model results found")

if ic_ft:
    rows.append(summarise(ic_ft, "Kronos Fine-tuned (LoRA)"))
else:
    print("⚠️  No fine-tuned results found")

if rows:
    df = pd.DataFrame(rows).set_index("Model").drop(columns="N (days)")
    print("\n" + "="*75)
    print("KRONOS BASE vs. FINE-TUNED  |  context=80, forecast=12  |  5 Seeds  |  2021+")
    print("="*75)
    print(df.to_string())
    print()

    if len(rows) == 2:
        delta = rows[1]["Mean RankIC"] - rows[0]["Mean RankIC"]
        sign  = "+" if delta >= 0 else ""
        direction = "↑ improvement" if delta > 0 else "↓ degradation"
        print(f"  ΔRankIC (Fine-tuned − Base) = {sign}{delta:.4f}  {direction}")

Loading Base-Model results …
  ⚠️  Missing: 01_model_comparison/results/kronos/seed_13/final_energy_study.json
  ⚠️  Missing: 01_model_comparison/results/kronos/seed_42/final_energy_study.json
  ⚠️  Missing: 01_model_comparison/results/kronos/seed_123/final_energy_study.json
  ⚠️  Missing: 01_model_comparison/results/kronos/seed_456/final_energy_study.json
  ⚠️  Missing: 01_model_comparison/results/kronos/seed_789/final_energy_study.json
Loading Fine-tuned results …


/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


⚠️  No base-model results found

KRONOS BASE vs. FINE-TUNED  |  context=80, forecast=12  |  5 Seeds  |  2021+
                          Mean RankIC            95% CI    Std  t-stat  p-value      Positive Days
Model                                                                                             
Kronos Fine-tuned (LoRA)       0.0254  [0.0091, 0.0418]  0.252    3.05   0.0024  3317/6120 (54.2%)



## 7 · Download Results

In [18]:
from google.colab import files
import zipfile
from pathlib import Path

zip_name = "kronos_finetuned_eval_5seeds.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    results_dir = Path("02_finetuning/results/kronos_finetuned")
    for f in results_dir.rglob("*.json"):
        zf.write(f, arcname=str(f.relative_to(results_dir)))

size_mb = Path(zip_name).stat().st_size / 1024 / 1024
print(f"Archive: {zip_name}  ({size_mb:.1f} MB)")
files.download(zip_name)
print("Downloaded.")

Archive: kronos_finetuned_eval_5seeds.zip  (3.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded.
